# Notebook 02 — Data Processing

This notebook processes raw data into analysis-ready intermediate datasets:
1. **Climate data:** ERA5-Land hourly grids → daily regional Tmax/Tmin/Tmean
2. **Mortality data:** ISTAT/Eurostat raw → NUTS-2 summer aggregates
3. **Socioeconomic data:** Raw indicators → merged panel

**Prerequisites:** Run Notebook 01 first. Raw data must be in `data/raw/`.

In [ ]:
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.utils.config import load_config, get_path

cfg = load_config()
print("Configuration loaded.")

---
## 1. Process Climate Data

Convert ERA5-Land hourly NetCDF files to daily regional averages:
- Compute daily **Tmax, Tmin, Tmean** from hourly T2m
- Compute daily mean **dewpoint temperature** (for humidity)
- Area-average grid cells within each NUTS-2 polygon

**Output:** `data/interim/daily_regional_climate.parquet`

In [ ]:
# First, inspect a sample ERA5-Land file to understand its structure
import xarray as xr

climate_dir = get_path(cfg, 'raw_data') / 'climate'
nc_files = sorted(climate_dir.glob('*.nc'))

if nc_files:
    sample_file = nc_files[0]
    print(f"Inspecting: {sample_file.name}\n")
    
    ds = xr.open_dataset(sample_file)
    print(ds)
    print(f"\nVariables: {list(ds.data_vars)}")
    print(f"Coordinates: {list(ds.coords)}")
    print(f"Time range: {ds.valid_time.values[0]} → {ds.valid_time.values[-1]}")
    ds.close()
else:
    print("No ERA5-Land files found. Download them first (see Notebook 01).")

In [ ]:
# Step-by-step: compute daily stats from hourly data
from src.utils.constants import KELVIN_OFFSET

if nc_files:
    ds = xr.open_dataset(nc_files[0])
    
    # Convert Kelvin to Celsius
    t2m_c = ds['t2m'] - KELVIN_OFFSET
    
    # Compute daily statistics
    daily_tmax = t2m_c.resample(valid_time='1D').max()
    daily_tmin = t2m_c.resample(valid_time='1D').min()
    daily_tmean = t2m_c.resample(valid_time='1D').mean()
    
    print(f"Daily Tmax shape: {daily_tmax.shape}")
    print(f"Date range: {daily_tmax.valid_time.values[0]} → {daily_tmax.valid_time.values[-1]}")
    print(f"Spatial grid: {daily_tmax.shape[1]} lat × {daily_tmax.shape[2]} lon")
    
    # Show a sample map
    fig, ax = plt.subplots(figsize=(8, 6))
    daily_tmax.isel(valid_time=30).plot(ax=ax, cmap='YlOrRd')
    ax.set_title(f'Daily Tmax — {str(daily_tmax.valid_time.values[30])[:10]}')
    plt.tight_layout()
    plt.show()
    
    ds.close()

In [ ]:
# Run the full climate processing pipeline
# This processes all years and aggregates to NUTS-2 regions

# ⚠️ This step can take 10-30 minutes depending on data size

# from src.data.process_climate import process_climate_data
# daily_climate = process_climate_data(cfg)
# print(f"Processed {len(daily_climate)} daily records")
# daily_climate.head()

In [ ]:
# If already processed, load the interim data
climate_path = get_path(cfg, 'interim_data') / 'daily_regional_climate.parquet'

if climate_path.exists():
    daily_climate = pd.read_parquet(climate_path)
    daily_climate['date'] = pd.to_datetime(daily_climate['date'])
    
    print(f"Shape: {daily_climate.shape}")
    print(f"Date range: {daily_climate['date'].min()} → {daily_climate['date'].max()}")
    print(f"Regions: {daily_climate['nuts2_code'].nunique()}")
    print(f"\nColumns: {daily_climate.columns.tolist()}")
    print(f"\nSample:")
    display(daily_climate.head(10))
else:
    print(f"Climate data not yet processed. Uncomment the cell above to run.")

---
## 2. Process Mortality Data

Load ISTAT or Eurostat mortality data and aggregate to NUTS-2 summer seasonal level.

**Expected input format:** CSV or Excel with columns for date, region, age group, deaths.

**Output:** `data/interim/mortality_processed.parquet`

In [ ]:
# Check what mortality files are available
mort_dir = get_path(cfg, 'raw_data') / 'mortality'
mort_files = [f for f in mort_dir.iterdir() if f.name != '.gitkeep']

if mort_files:
    print(f"Found {len(mort_files)} mortality files:")
    for f in mort_files:
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  {f.name}  ({size_mb:.1f} MB)")
    
    # Preview the first file
    f = mort_files[0]
    if f.suffix == '.csv':
        preview = pd.read_csv(f, nrows=5)
    elif f.suffix == '.xlsx':
        preview = pd.read_excel(f, nrows=5)
    else:
        preview = None
    
    if preview is not None:
        print(f"\nPreview of {f.name}:")
        print(f"Columns: {preview.columns.tolist()}")
        display(preview)
else:
    print("No mortality data found. See data_download_guide.md for instructions.")

In [ ]:
# Process mortality data (uncomment when files are available)

# from src.data.process_mortality import process_mortality_data
#
# mortality = process_mortality_data(cfg)
# if not mortality.empty:
#     print(f"Processed mortality data: {mortality.shape}")
#     display(mortality.head())
# else:
#     print("No mortality data processed — check column format.")

---
## 3. Process Socioeconomic Data

Load and merge socioeconomic indicators for RSVI construction:
- Age structure (% population 65+, 75+, 80+)
- Economic indicators (poverty rate, GDP per capita)
- Urban density (population density, urbanization rate)

**Output:** `data/interim/socioeconomic_processed.parquet`

In [ ]:
# Check what socioeconomic files are available
socio_dir = get_path(cfg, 'raw_data') / 'socioeconomic'
socio_files = [f for f in socio_dir.iterdir() if f.name != '.gitkeep']

if socio_files:
    print(f"Found {len(socio_files)} socioeconomic files:")
    for f in socio_files:
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  {f.name}  ({size_mb:.1f} MB)")
        
        # Preview
        try:
            if f.suffix == '.csv':
                preview = pd.read_csv(f, nrows=3)
            elif f.suffix == '.xlsx':
                preview = pd.read_excel(f, nrows=3)
            else:
                continue
            print(f"  Columns: {preview.columns.tolist()}")
        except Exception as e:
            print(f"  Error reading: {e}")
else:
    print("No socioeconomic data found. See data_download_guide.md for instructions.")

In [ ]:
# Process socioeconomic data (uncomment when files are available)

# from src.data.process_socioeconomic import process_socioeconomic_data
#
# socio = process_socioeconomic_data(cfg)
# if not socio.empty:
#     print(f"Processed socioeconomic data: {socio.shape}")
#     display(socio.head())
# else:
#     print("No socioeconomic data processed.")

---
## Summary of Processed Data

Check which interim datasets have been created.

In [ ]:
interim_dir = get_path(cfg, 'interim_data')

expected_files = [
    'daily_regional_climate.parquet',
    'mortality_processed.parquet',
    'socioeconomic_processed.parquet',
]

print("Interim data status:")
for fname in expected_files:
    path = interim_dir / fname
    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"  ✅ {fname}  ({size_mb:.1f} MB)")
    else:
        print(f"  ❌ {fname}  (not yet created)")

print("\n➡️ Next: Notebook 03 — Feature Engineering")